In [1]:
import os
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple

from src.config import MODEL_CLASSES
import src.api.result as result_api


RESULTS_DIR = os.path.join("results")
FIGURES_DIR = os.path.join("figures")

os.makedirs(FIGURES_DIR, exist_ok=True)

# Shuffle

In [2]:
def load_results(model_name, component, type):
    fname = f"{model_name}_symbol_20_shuffle_{component}.json"
    result_path = os.path.join(RESULTS_DIR, fname)
    with open(result_path, "r") as f:
        results = json.load(f)

    results = [result_api.ICLResult(**r) for r in results]

    # Initialize dictionary to store probabilities
    d = {
        "original": [],
        "inside": {},
        "outside": {},
    }

    for r in results:
        if "shuffle_meta" in r.model_details:
            shuffle_meta = r.model_details["shuffle_meta"]
            shuffle_type = shuffle_meta["shuffled_type"]
            shuffle_seed = str(shuffle_meta["seed"])
            d[shuffle_type][shuffle_seed] = r.acc if type == "acc" else r.prob
        else:
            d["original"] = r.acc if type == "acc" else r.prob

    return d

In [18]:
d_qk_acc, d_ov_acc = {}, {}
d_qk_prob, d_ov_prob = {}, {}
for model_name in MODEL_CLASSES:
    try:
        d_qk_acc[model_name] = load_results(model_name, "qk", "acc")
        d_ov_acc[model_name] = load_results(model_name, "ov", "acc")
        d_qk_prob[model_name] = load_results(model_name, "qk", "prob")
        d_ov_prob[model_name] = load_results(model_name, "ov", "prob")
    except FileNotFoundError:
        print(f"File not found for {model_name}")
        continue


with open("shuffle_results.json", "w") as f:
    json.dump(
        {
            "qk_acc": d_qk_acc,
            "ov_acc": d_ov_acc,
        },
        f,
    )

File not found for pythia-14m
File not found for pythia-36m
File not found for pythia-70m
File not found for pythia-160m
File not found for pythia-410m
File not found for pythia-1b
File not found for pythia-1_4b
File not found for pythia-2_8b


In [21]:
def aggregate_shuffle_results(d_qk, d_ov, type, seed=0):
    cmap = matplotlib.colormaps.get_cmap("brg")
    colors = [cmap(t) for t in np.linspace(0, 0.9, len(d_qk))]

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    label_handle = []
    label_names = []
    for i, model_name in enumerate(sorted(d_qk.keys())):
        d_model = d_qk[model_name]
        mu_original = d_model["original"]
        mu_inside = d_model["inside"][str(seed)]
        mu_outside = d_model["outside"][str(seed)]
        p1 = ax.scatter(
            mu_original - mu_inside,
            mu_original - mu_outside,
            marker="+",
            color=colors[i],
            s=100,
            label=model_name + " qk",
        )

        d_model = d_ov[model_name]
        mu_original = d_model["original"]
        mu_inside = d_model["inside"][str(seed)]
        mu_outside = d_model["outside"][str(seed)]
        p2 = ax.scatter(
            (mu_original - mu_inside) / mu_original,
            (mu_original - mu_outside) / mu_original,
            marker="x",
            color=colors[i],
            s=100,
            label=model_name + " ov",
        )
        label_handle.append((p1, p2))
        label_names.append(model_name + " QK/OV")

    ax.add_line(plt.Line2D([0, 1], [0, 1], color="black", linestyle="--"))
    ax.set_xlim((0 - 0.05, 1))
    ax.set_ylim((0 - 0.05, 1))
    l = ax.legend(
        label_handle, label_names, handler_map={tuple: HandlerTuple(ndivide=None)}
    )
    title = "Reduced acc of predicting correct token"
    ax.set_title(title, weight="bold", fontsize=12)
    ax.set_xlabel("Shuffle heads within", weight="bold", fontsize=14)
    ax.set_ylabel("Shuffle heads outside", weight="bold", fontsize=14)
    plt.savefig(
        os.path.join(FIGURES_DIR, f"shuffle_summary_{type}_seed{seed}.png"),
        bbox_inches="tight",
    )
    plt.close()

In [22]:
for type in ["prob", "acc"]:
    for seed in [0, 20, 40, 60, 80]:
        aggregate_shuffle_results(d_qk_prob, d_ov_prob, type=type, seed=seed)

In [6]:
def load_details_probs(model_name, component):
    fname = f"{model_name}_symbol_20_shuffle_{component}.json"
    result_path = os.path.join(RESULTS_DIR, fname)
    with open(result_path, "r") as f:
        results = json.load(f)

    results = [result_api.ICLResult(**r) for r in results]

    d = {"original": [], "inside": {}, "outside": {}}

    for r in results:
        probs = [example["prob"] for example in r.examples]
        if "shuffle_meta" in r.model_details:
            shuffle_meta = r.model_details["shuffle_meta"]
            shuffle_type = shuffle_meta["shuffled_type"]
            shuffle_seed = str(shuffle_meta["seed"])
            d[shuffle_type][shuffle_seed] = np.array(probs).flatten()
        else:
            d["original"] = np.array(probs).flatten()

    return d


d_qk, d_ov = {}, {}
for model_name in MODEL_CLASSES:
    try:
        d_qk[model_name] = load_details_probs(model_name, "qk")
        d_ov[model_name] = load_details_probs(model_name, "ov")
    except FileNotFoundError:
        print(f"File not found for {model_name}")
        continue

File not found for pythia-14m
File not found for pythia-36m
File not found for pythia-70m
File not found for pythia-160m
File not found for pythia-410m
File not found for pythia-1b
File not found for pythia-1_4b
File not found for pythia-2_8b


In [7]:
def shuffle_to_hist(d, model_name, circuit):
    labels = ["original model", "shuffle within", "shuffle outside"]
    seeds = [0, 20, 40, 60, 80]
    fig, axs = plt.subplots(len(seeds), 1, figsize=(8, 8))
    for i, seed in enumerate(seeds):
        counts = [
            d[model_name]["original"],
            d[model_name]["inside"][str(seed)],
            d[model_name]["outside"][str(seed)],
        ]

        bins = np.linspace(0, 1, 25)
        axs[i].hist(counts, bins, density=True, label=labels)
        if i == 2:
            xlabel = "Prob of predicting correct token"
            axs[i].set_xlabel(xlabel, weight="bold", fontsize=14)
        if i == 0:
            axs[i].legend(prop={"size": 12, "weight": "bold"})
            axs[i].set_title(circuit, weight="bold")
        if i != 2:
            axs[i].set_xticks([])

    fig.text(
        0.03,
        0.5,
        "Density histogram",
        ha="center",
        va="center",
        rotation="vertical",
        weight="bold",
        fontsize=14,
    )
    plt.savefig(
        os.path.join(FIGURES_DIR, f"shuffle_hist_{model_name}_{circuit}.png"),
        bbox_inches="tight",
    )
    plt.close()

In [8]:
for d, circuit in zip([d_qk, d_ov], ["QK", "OV"]):
    for model_name in d.keys():
        shuffle_to_hist(d, model_name, circuit)

# Project

In [23]:
from src.evaluate_projection import ProjectionResult


def load_results_projection(model_name, component, type):
    fname = f"{model_name}_symbol_20_projection_{component}.json"
    result_path = os.path.join(RESULTS_DIR, fname)
    with open(result_path, "r") as f:
        results = json.load(f)

    r_true, r_false = [ProjectionResult(**r) for r in results]

    d_model = r_true.model_details["hidden_size"]
    K = max(50, int(0.05 * d_model / 10) * 10)
    rank = r_true.result["rank"]
    proj_true = r_true.result["acc"] if type == "acc" else r_true.result["prob"]
    proj_false = r_false.result["acc"] if type == "acc" else r_false.result["prob"]

    return {
        "original": proj_true[0],
        "K": K,
        "proj_true_K": proj_true[rank.index(K)],
        "proj_false_K": proj_false[rank.index(K)],
    }

In [24]:
d_qk_acc = {}
d_qk_prob = {}
for model_name in MODEL_CLASSES:
    try:
        d_qk_acc[model_name] = load_results_projection(model_name, "qk", "acc")
        d_qk_prob[model_name] = load_results_projection(model_name, "qk", "prob")
    except FileNotFoundError:
        print(f"File not found for {model_name}")
        continue

File not found for pythia-14m
File not found for pythia-36m
File not found for pythia-70m
File not found for pythia-160m
File not found for pythia-410m
File not found for pythia-1b
File not found for pythia-1_4b
File not found for pythia-2_8b


In [27]:
def plot_projection_results(d, component, type):
    import matplotlib
    from matplotlib.legend_handler import HandlerTuple

    cmap = matplotlib.colormaps.get_cmap("brg")
    colors = [cmap(t) for t in np.linspace(0, 0.9, len(d))]

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    label_handle = []
    label_names = []

    for i, model_name in enumerate(sorted(d.keys())):
        baseline = d[model_name]["original"]
        proj_true = d[model_name]["proj_true_K"]
        proj_false = d[model_name]["proj_false_K"]

        p1 = ax.scatter(
            (baseline - proj_true) / baseline,
            (baseline - proj_false) / baseline,
            marker="+",
            s=100,
            color=colors[i],
        )
        label_handle.append((p1))
        label_names.append(model_name)

    ax.add_line(plt.Line2D([0, 1], [0, 1], color="black", linestyle="--"))
    ax.set_xlim((0 - 0.05, 1 + 0.05))
    ax.set_ylim((0 - 0.05, 1 + 0.05))
    l = ax.legend(
        label_handle, label_names, handler_map={tuple: HandlerTuple(ndivide=None)}
    )
    title = "Reduced acc under edited model"
    ax.set_title(title, weight="bold", fontsize=13)
    ax.set_xlabel("Remove subspace", weight="bold", fontsize=14)
    ax.set_ylabel("Keep subspace", weight="bold", fontsize=14)
    plt.savefig(
        os.path.join(FIGURES_DIR, f"projection_summary_{component}_{type}.png"),
        bbox_inches="tight",
    )
    plt.close()

In [28]:
for d, type in zip([d_qk_acc, d_qk_prob], ["acc", "prob"]):
    plot_projection_results(d, "qk", type)